# Treino na GPU do Kaggle: a formulação de sucessor e a curva de aprendizado

Roda o que é pesado demais para a RTX 4050 de 6 GB, com o **mesmo protocolo** do projeto
(mesmas páginas, mesma referência, blocos embaralhados pela mesma permutação):

1. **curva de aprendizado** das duas formulações do LayoutLM — a pontual (um escore por bloco)
   e a de **sucessor** (prevê qual bloco vem depois), em 5 tamanhos de treino;
2. **ablações** da formulação de sucessor: sem o viés geométrico, sem a perda auxiliar de
   posição, sem o layout 2D;
3. **varredura do limiar de decodificação**, que regula quanto o modelo confia nas cadeias
   locais antes de recorrer à ordem global;
4. **latência de todos os métodos no mesmo hardware**, para a comparação de eficiência ser justa;
5. **avaliação contra a anotação humana**, se o lote do Golden Dataset já estiver anotado.

**Antes de rodar**
1. *Settings* → *Accelerator*: **GPU T4 x2** · *Internet*: **On** (o LayoutLM base vem do Hugging Face).
2. *Add Input* → o dataset com `tcc_ordem_leitura.zip` (gerado por `src/kaggle/montar_pacote.py`).
3. *Save Version* → **Save & Run All (Commit)**: roda em segundo plano por até 12 h, com o navegador fechado.
4. No fim, baixe `resultados_kaggle.zip` na aba *Output* e rode no projeto
   `venv\Scripts\python.exe src\kaggle\importar_resultados.py resultados_kaggle.zip`.

In [ ]:
# ===== CONFIGURACAO =====
TAMANHOS = [152, 400, 1000, 2000, 0]     # paginas de treino; 0 = todas as extras disponiveis (3.235)
MODELOS = ["layoutlm_sucessor", "layoutlm"]   # a formulacao nova primeiro, caso o tempo acabe
MICRO_BATCH = 12         # a T4 tem 16 GB; o batch EFETIVO continua 8 (acumulacao), so muda a velocidade
ABLACOES = True          # sem vies geometrico / sem perda de posicao / sem layout 2D
LIMIARES = [0.0, 0.3, 0.5, 0.7, 0.9]     # varredura do limiar da cadeia (so avaliacao, custa segundos)
LOTE_GOLDEN = "golden_v1"
GUARDAR_CHECKPOINT = True                # salva o melhor modelo em /kaggle/working para baixar (~450 MB)

In [ ]:
import os, sys, glob, json, time, shutil, subprocess, urllib.request

achados = glob.glob("/kaggle/input/**/data/processed/splits.json", recursive=True)
assert achados, "Pacote nao encontrado: use Add Input e adicione o dataset com tcc_ordem_leitura.zip"
origem = os.path.dirname(os.path.dirname(os.path.dirname(achados[0])))
PROJ = "/kaggle/working/tcc"
if not os.path.exists(PROJ):
    shutil.copytree(origem, PROJ)
os.chdir(PROJ)
os.makedirs("models/layoutlm_ordem", exist_ok=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)

def rodar(cmd, mostrar=True, ultimas_n=3):
    if isinstance(cmd, str):
        cmd = cmd.split()
    print("$", " ".join(str(c) for c in cmd), flush=True)
    t0 = time.time()
    proc = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    ultimas = []
    for linha in proc.stdout:
        if mostrar:
            print(linha, end="", flush=True)
        ultimas = (ultimas + [linha])[-ultimas_n:]
    codigo = proc.wait()
    if not mostrar:
        print("".join(ultimas).strip()[-500:])
    print(f"  [{time.time() - t0:.0f} s, codigo {codigo}]", flush=True)
    return codigo

print("Projeto em", PROJ)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Latencia de ingenua, XY-cut, BiLSTM e LayoutLM NESTA maquina, com o mesmo roteiro de medicao
rodar([sys.executable, "src/eval/latencia_hardware.py", "--maquina", "kaggle_t4"])

In [ ]:
# Curva de aprendizado das duas formulacoes. E a parte longa: ~1 h por modelo na T4.
rodar([sys.executable, "src/rodar_curva_aprendizado.py", "--modelos", *MODELOS, "--tamanhos", *TAMANHOS,
       "--sem-lstm", "--micro-batch", MICRO_BATCH])
print(open("resultados/curva_aprendizado/curva.json").read()[:400])

In [ ]:
# O maior treino que a curva de fato usou, e o checkpoint que ele deixou
N_MAIOR = json.load(open("resultados/curva_aprendizado/curva.json"))["tamanhos"][-1]
CKPT = f"curva_layoutlm_sucessor_n{N_MAIOR}"
if not os.path.exists(f"models/layoutlm_ordem/{CKPT}.pt"):
    CKPT = "layoutlm_sucessor"      # a curva nao chegou ao fim; cai para o treino padrao
print("maior treino:", N_MAIOR, "paginas | checkpoint:", CKPT)

In [ ]:
# Ablacoes da formulacao de sucessor, no maior treino: o que cada peca do modelo contribui
VARIANTES = [(["--sem-vies-geo"], "ablacao_suc_sem_vies_geo", "sem o MLP de geometria relativa"),
             (["--peso-posicao", "0"], "ablacao_suc_sem_posicao", "sem a perda auxiliar de posicao"),
             (["--sem-layout"], "ablacao_suc_sem_layout", "sem as bounding boxes (so texto)")]
if ABLACOES:
    for flags, nome, o_que in VARIANTES:
        print("\n===== ablacao:", o_que, "=====", flush=True)
        base = [sys.executable, "src/models/layoutlm_sucessor.py"]
        if rodar(base + ["treinar", "--nome", nome, "--n-treino", N_MAIOR, *flags,
                         "--blocos-extra", "data/processed/blocos_extra", "--epocas", "4",
                         "--max-minutos", "60", "--micro-batch", MICRO_BATCH], mostrar=False) == 0:
            rodar(base + ["avaliar", "--nome", nome, *flags], mostrar=False)

In [ ]:
# Varredura do limiar: quanto o modelo deve confiar nas cadeias locais antes de recorrer
# a ordem global? Custa segundos, porque so troca a decodificacao de um modelo ja treinado.
for lim in LIMIARES:
    rodar([sys.executable, "src/models/layoutlm_sucessor.py", "avaliar", "--nome", CKPT,
           "--decodificacao", "cadeias", "--limiar-cadeia", lim,
           "--metodo", "suc_limiar_" + str(lim).replace(".", "")], mostrar=False)
rodar([sys.executable, "src/models/layoutlm_sucessor.py", "avaliar", "--nome", CKPT,
       "--decodificacao", "feixe", "--metodo", "layoutlm_sucessor_feixe"], mostrar=False)

In [ ]:
# Contra a anotacao humana, se o lote ja estiver anotado
if os.path.exists(f"data/golden/{LOTE_GOLDEN}/anotacoes.jsonl"):
    rodar([sys.executable, "src/eval/avaliar_golden.py", "--lote", LOTE_GOLDEN,
           "--layoutlm", "layoutlm", "--sucessor", CKPT])
else:
    print(f"Lote {LOTE_GOLDEN} ainda sem anotacoes.jsonl -- anote no projeto e refaca o pacote.")

In [ ]:
# Tabela final com tudo o que existe: baselines, as duas formulacoes, ablacoes e limiares
rodar([sys.executable, "src/eval/comparar.py"])
rodar([sys.executable, "src/rodar_curva_aprendizado.py", "--so-figura"], mostrar=False)

In [ ]:
import zipfile
destino = "/kaggle/working/resultados_kaggle.zip"
with zipfile.ZipFile(destino, "w", zipfile.ZIP_DEFLATED) as z:
    for arq in glob.glob("resultados/**/*", recursive=True):
        if os.path.isfile(arq) and not arq.endswith(".pt"):
            z.write(arq)
    for arq in glob.glob("models/layoutlm_ordem/*_config.json"):
        z.write(arq)
    for arq in glob.glob("docs/figuras/*.png"):
        z.write(arq)
    for arq in glob.glob("/kaggle/working/*.log"):
        z.write(arq, "logs/" + os.path.basename(arq))
print("Pronto:", destino, f"({os.path.getsize(destino) / 2**20:.1f} MB). Baixe na aba Output.")
if GUARDAR_CHECKPOINT:
    for arq in sorted(glob.glob("models/layoutlm_ordem/curva_layoutlm_sucessor_n*.pt")):
        shutil.copy(arq, "/kaggle/working/" + os.path.basename(arq))
        print("checkpoint para baixar:", os.path.basename(arq),
              f"({os.path.getsize(arq) / 2**20:.0f} MB)")